# Customer Support Router | Routing

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class RouterState(TypedDict):
    input: str
    category: str
    response: str

In [5]:
# Router node: classify the input
def classify_input(state: RouterState) -> dict:
    response = model.invoke(
        f"Classify this query into exactly one category: 'billing', 'technical', or 'general'.\n"
        f"Query: {state['input']}\n"
        f"Respond with only the category name."
    )
    return {"category": response.content.strip().lower()}

# Routing logic (exact match with fallback to general)
def route_by_category(state: RouterState) -> Literal["billing_agent", "technical_agent", "general_agent"]:
    routing_map = {
        "billing": "billing_agent",
        "technical": "technical_agent",
        "general": "general_agent",
    }
    category = state["category"].strip().lower()
    return routing_map.get(category, "general_agent")

# Specialized handlers
def billing_agent(state: RouterState) -> dict:
    response = model.invoke(
        f"You are a billing specialist. Help with: {state['input']}"
    )
    return {"response": response.content}

def technical_agent(state: RouterState) -> dict:
    response = model.invoke(
        f"You are a technical support expert. Help with: {state['input']}"
    )
    return {"response": response.content}

def general_agent(state: RouterState) -> dict:
    response = model.invoke(
        f"You are a helpful assistant. Help with: {state['input']}"
    )
    return {"response": response.content}

In [6]:
# Build the graph
graph = StateGraph(RouterState)
graph.add_node("classifier", classify_input)
graph.add_node("billing_agent", billing_agent)
graph.add_node("technical_agent", technical_agent)
graph.add_node("general_agent", general_agent)

graph.add_edge(START, "classifier")
graph.add_conditional_edges("classifier", route_by_category)
graph.add_edge("billing_agent", END)
graph.add_edge("technical_agent", END)
graph.add_edge("general_agent", END)

router = graph.compile()

In [7]:
# Plot the workflow
plot_mermaid(router)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classifier(classifier)
	billing_agent(billing_agent)
	technical_agent(technical_agent)
	general_agent(general_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> classifier;
	classifier -.-> billing_agent;
	classifier -.-> general_agent;
	classifier -.-> technical_agent;
	billing_agent --> __end__;
	general_agent --> __end__;
	technical_agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = router.invoke({"input": "I was charged twice on my last invoice"})
print(result["response"])

I'm sorry to hear about the double charge on your last invoice. Here’s a step-by-step guide to address the issue:

1. **Review the Invoice:**
   - Verify the details on your invoice to confirm the duplicate charge.
   - Take note of the dates, amounts, and any reference numbers associated with the charges.

2. **Check Payment Records:**
   - Look into your bank or credit card statements to ensure that both charges were processed.

3. **Gather Documentation:**
   - Prepare any relevant documents, such as copies of the invoice, payment confirmation, and bank statements showing the double charge.

4. **Contact Customer Service:**
   - Reach out to the billing or customer service department of the company.
   - You can usually find contact details on the invoice or the company's website.

5. **Explain the Situation:**
   - Clearly explain the error and provide the details you gathered.
   - Reference invoice numbers, dates, and any other relevant information to support your claim.

6. **Re

In [13]:
# Streaming

output = stream_invoke(router, {"input": "I was charged twice on my last invoice"})
output


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'input': 'I was charged twice on my last invoice',
 'category': 'billing',
 'response': "I'm sorry to hear about the double charge on your last invoice. Here’s a step-by-step guide to address the issue:\n\n1. **Review the Invoice:**\n   - Verify the details on your invoice to confirm the duplicate charge.\n   - Take note of the dates, amounts, and any reference numbers associated with the charges.\n\n2. **Check Payment Records:**\n   - Look into your bank or credit card statements to ensure that both charges were processed.\n\n3. **Gather Documentation:**\n   - Prepare any relevant documents, such as copies of the invoice, payment confirmation, and bank statements showing the double charge.\n\n4. **Contact Customer Service:**\n   - Reach out to the billing or customer service department of the company.\n   - You can usually find contact details on the invoice or the company's website.\n\n5. **Explain the Situation:**\n   - Clearly explain the error and provide the details you gathered

In [14]:
print(output['response'])

I'm sorry to hear about the double charge on your last invoice. Here’s a step-by-step guide to address the issue:

1. **Review the Invoice:**
   - Verify the details on your invoice to confirm the duplicate charge.
   - Take note of the dates, amounts, and any reference numbers associated with the charges.

2. **Check Payment Records:**
   - Look into your bank or credit card statements to ensure that both charges were processed.

3. **Gather Documentation:**
   - Prepare any relevant documents, such as copies of the invoice, payment confirmation, and bank statements showing the double charge.

4. **Contact Customer Service:**
   - Reach out to the billing or customer service department of the company.
   - You can usually find contact details on the invoice or the company's website.

5. **Explain the Situation:**
   - Clearly explain the error and provide the details you gathered.
   - Reference invoice numbers, dates, and any other relevant information to support your claim.

6. **Re